# **# 468Page 스팸 메일 분류하기**
**202235203 고수호**


# **# 필요 라이브러리 임포트**

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# **# 데이터 다운로드**

In [4]:
imdb = keras.datasets.imdb
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


# **# 이터와 테스트 데이터의 길이를 튜플로 출력**

In [5]:
len(x_train), len(x_test)

(25000, 25000)

# **# 첫 번째 데이터 출력**

In [6]:
print(x_train[0])

[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]


# **# 데이터 길이 출력**

In [7]:
len(x_train[0]), len(x_train[1])

(218, 189)

# **# 레이블 출력**

In [8]:
y_train[0], y_train[1]

(np.int64(1), np.int64(0))

# **# 긍정적인 리뷰와 부정적인 리뷰 개수 출력**

In [9]:
np.unique(y_train, return_counts=True)

(array([0, 1]), array([12500, 12500]))

# **# 각 단어와 인덱스가 저장된 딕셔너리 반환**

In [10]:
word_to_index = imdb.get_word_index()
word_to_index = {k:(v+3) for k,v in word_to_index.items()}
word_to_index["<PAD>"] = 0
word_to_index["<START>"] = 1
word_to_index["<UNK>"] = 2
word_to_index["<UNUSED>"] = 3

index_to_word = dict([(value, key) for (key, value) in word_to_index.items()])

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [11]:
print(' '.join([index_to_word[index] for index in x_train[0]]))

<START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for <UNK> and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also <UNK> to the two little boy's that played the <UNK> of norman and paul they were just brilliant children are often left out of the <UNK> list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for wha

# **# 필요 라이브러리 임포트**

In [12]:
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import *

# **# 크가가 제각각이므로 일정 크기 이하로 제한**

In [13]:
x_train = pad_sequences(x_train, maxlen=100)
x_test = pad_sequences(x_test, maxlen=100)

# **# 학습 예제의 길이 확인**

In [14]:
len(x_train[0]), len(x_train[1])

(100, 100)

# **# 첫 번째 영화 리뷰 출력**

In [15]:
print(x_train[0])

[1415   33    6   22   12  215   28   77   52    5   14  407   16   82
    2    8    4  107  117 5952   15  256    4    2    7 3766    5  723
   36   71   43  530  476   26  400  317   46    7    4    2 1029   13
  104   88    4  381   15  297   98   32 2071   56   26  141    6  194
 7486   18    4  226   22   21  134  476   26  480    5  144   30 5535
   18   51   36   28  224   92   25  104    4  226   65   16   38 1334
   88   12   16  283    5   16 4472  113  103   32   15   16 5345   19
  178   32]


# **# 신경망 구축**

In [16]:
vocab_size = 10000

model = Sequential()
model.add(Embedding(vocab_size, 64, input_length=100))
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [17]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
history = model.fit(x_train, y_train, epochs=20, batch_size=54, verbose=1, validation_data=(x_test, y_test))

Epoch 1/20
463/463 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - accuracy: 0.7699 - loss: 0.4532 - val_accuracy: 0.8478 - val_loss: 0.3424
Epoch 2/20
463/463 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - accuracy: 0.9391 - loss: 0.1684 - val_accuracy: 0.8302 - val_loss: 0.4173
Epoch 3/20
463/463 ━━━━━━━━━━━━━━━━━━━━ 14s 30ms/step - accuracy: 0.9928 - loss: 0.0293 - val_accuracy: 0.8316 - val_loss: 0.5587
Epoch 4/20
463/463 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - accuracy: 0.9989 - loss: 0.0062 - val_accuracy: 0.8362 - val_loss: 0.6177
Epoch 5/20
463/463 ━━━━━━━━━━━━━━━━━━━━ 11s 25ms/step - accuracy: 0.9995 - loss: 0.0031 - val_accuracy: 0.8356 - val_loss: 0.6788
Epoch 6/20
463/463 ━━━━━━━━━━━━━━━━━━━━ 11s 24ms/step - accuracy: 1.0000 - loss: 8.2371e-04 - val_accuracy: 0.8383 - val_loss: 0.7209
Epoch 7/20
463/463 ━━━━━━━━━━━━━━━━━━━━ 11s 24ms/step - accuracy: 1.0000 - loss: 4.3126e-04 - val_accuracy: 0.8386 - val_loss: 0.7545
Epoch 8/20
463/463 ━━━━━━━━━━━━━━━━━━━━ 13s 28ms/step - accuracy: 1.0000 - loss: 2

# **# 모델 성능 평가**

In [18]:
results = model.evaluate(x_test, y_test, verbose=2)
print(results)

782/782 - 2s - 3ms/step - accuracy: 0.8264 - loss: 1.1657
[1.1656700372695923, 0.8264399766921997]


# **# 직접 작성한 리뷰로 테스트**

In [19]:
review = "What can I say about this movie that was already said? It is my favorite time travel sci-fi, adventure epic comedy in the 80's and I love this movie to death!"

# **# 알파벳만 남기고 나머지 특수 문자들은 전부 삭제**

In [20]:
import re
review = re.sub("[^0-9a-zA-Z]", "", review).lower()

# **# 단어를 하나씩 꺼내서 정수 인덱스로 변환**

In [22]:
review_encoding = []
for w in review.split():
  index = word_to_index.get(w, 2)
  if index <= 10000:
    review_encoding.append(index)
  else:
    review_encoding.append(word_to_index["UNK"])

test_input = pad_sequences([review_encoding], maxlen = 100)
value = model.predict(test_input)
if (value > 0.5):
  print("긍정적인 리뷰입니다.")
else:
  print("부정적인 리뷰입니다.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step
부정적인 리뷰입니다.
